# Source audio analysis
This notebook reads the preserved Hugging Face source snapshot under `sources/`.
For model training, use the organized `train.csv` and `test.csv` at the dataset root; see `dataset_splits.ipynb`.

In [ ]:
from pathlib import Path
import wave

import pandas as pd
from datasets import Audio, Dataset

root = next((parent / "data" for parent in (Path.cwd(), *Path.cwd().parents)
             if (parent / "data/karakalpak-audio-dataset-hf/sources/data.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("karakalpak-audio-dataset-hf/sources/data.csv")
hf_dir = root / "karakalpak-audio-dataset-hf" / "sources"
frame = pd.read_csv(hf_dir / "data.csv")
if frame["audio_index"].tolist() != list(range(len(frame))):
    raise ValueError("data.csv audio_index must be sequential")
 
def sample_rate(path):
    with wave.open(str(path), "rb") as audio:
        return audio.getframerate()

data_files = sorted((hf_dir / "data").glob("*.wav"), key=lambda path: (sample_rate(path), path.name))
audio_16k_by_name = {path.name: path for path in (hf_dir / "data_16k").glob("*.wav")}
if frame["filename"].tolist() != [path.name for path in data_files] or set(frame["filename"]) != set(audio_16k_by_name):
    raise ValueError("data.csv rows must match both audio folders")
frame["audio_original"] = [{"path": str(path.resolve()), "bytes": None} for path in data_files]
frame["audio"] = [{"path": str(audio_16k_by_name[name].resolve()), "bytes": None} for name in frame["filename"]]
dataset = Dataset.from_pandas(frame, preserve_index=False).cast_column("audio_original", Audio(decode=False)).cast_column("audio", Audio(decode=False))
dataset

In [ ]:
print(dataset)

In [ ]:
import pandas as pd 
import numpy as np

In [ ]:
data = pd.read_csv(hf_dir / "data.csv")
data.head()